In [2]:
import torch
import numpy as np
import math
import matplotlib.pyplot as plt
import seaborn as sns
import torch.nn.functional as F
import random

In [3]:
words = open("names.txt" ,'r').read().splitlines()
len(words)
#vocaublary from words
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] =0
itos = {i:s for s,i in stoi.items()}
itos
voab_size = len(stoi)
print(voab_size)

27


In [4]:
block_size =3
def build_dataset(words):
    X,Y=[],[]

    for w in words:
        context = [0]*block_size       
        for i in w+'.':
            X.append([i for i in context])
            Y.append(stoi[i])
            context = context[1:]+[stoi[i]]
    
    x= torch.tensor(X)
    y=torch.tensor(Y)
    return x,y


# x,y =build_dataset(words[:3])
# print(x)
# print(len(y))   
random.shuffle(words)
n1 = int(0.8*len(words))
n2 = int(0.9*len(words))

xtr , ytr = build_dataset(words[:n1]) 
xval , yval = build_dataset(words[n1:n2]) 
xtest , ytest = build_dataset(words[n2:])     
        
    

In [15]:
n_embd =10 # dimension of embedding 
n_hidden =200 # neuron in hidden layer

g= torch.Generator().manual_seed(2147483647)
C = torch.randn((voab_size,n_embd) , generator=g)

w1 = torch.randn((n_embd*block_size,n_hidden) , generator=g )*(5/3)/(n_embd*block_size)**0.5
b1 = torch.randn(n_hidden , generator=g ) * 0.01
w2 = torch.randn((n_hidden,voab_size) , generator=g)*0.01
b2 = torch.randn(voab_size , generator=g )*0

bngain = torch.ones((1,n_hidden))
bnbias = torch.zeros((1,n_hidden))

bnmean_running = torch.zeros((1,n_hidden))
bnstd_running = torch.zeros((1,n_hidden))

parameters = [C,w1,w2,b1,b2 , bnbias,bngain]
number_of_parameters = sum(p.nelement() for p in parameters)
print(number_of_parameters)
for p in parameters:
    p.requires_grad =True
    
w1.shape

12297


torch.Size([30, 200])

In [16]:
# lossi =[]
# for i in range(max_step):
#     ix = torch.randint(0,xtr.shape[0],(batch_size,),generator=g)
#     xb,yb = xtr[ix],ytr[ix]
#     # forward pass
#     embd = C[xb]
#     embd_cat = embd.view(embd.shape[0],-1)
#     hpreact = embd_cat @ w1 +b1
#     #batch normilation
#     bnmean =hpreact.mean(dim=0 ,keepdim=True)
#     bnstd =hpreact.std(dim=0 ,keepdim=True)
#     hpreact = bngain *((hpreact -bnmean)/bnstd) + bnbias    
    
#     h = torch.tanh(hpreact)
#     logits = h@w2+b2
#     loss  = F.cross_entropy(logits,yb)
    
#     with torch.no_grad():
#         bnmean_running = 0.999*bnmean_running + 0.001*bnmean
#         bnstd_running = 0.999*bnstd_running + 0.001*bnstd
    
#     for p in parameters:
#         p.grad = None
#     loss.backward()
#     lr = 0.1 if i< 100000 else 0.01
#     for p in parameters:
#         p.data += -lr * p.grad
    
#     if i % 10000 ==0:
#         print(f'{i}/{max_step}->  loss: {loss.item()}')
#     lossi.append(loss.log10().item())

In [17]:
max_step = 200000
batch_size =32 
n= batch_size
ix = torch.randint(0,xtr.shape[0],(batch_size,),generator=g)
xb,yb = xtr[ix],ytr[ix]

embd = C[xb]
embd_cat = embd.view(embd.shape[0],-1)
hprebn = embd_cat @ w1 +b1
bnmean = 1/n * hprebn.sum(0,keepdim= True)
bndiff = hprebn - bnmean 
bndiff2 = bndiff**2
bnvarr = 1/(n-1) * (bndiff2).sum(0,keepdim =True)
bnvarr_inv = (bnvarr + 1e-15)**-0.5
bnraw = bndiff * bnvarr_inv
hpreact = bngain * bnraw + bnbias

h= torch.tanh(hpreact)

logits = h@ w2 + b2
logits_maxes = logits.max(1, keepdim=True).values
norm_logits = logits - logits_maxes 
counts  = norm_logits.exp()


counts_sum = counts.sum(1, keepdim = True)
counts_sum_invr = counts_sum**-1
probes = counts*counts_sum_invr

logprob = probes.log()


loss = -logprob[range(n),yb].mean()


#pytorch backward 

for p in parameters:
    p.grad = None

for t in [logprob,probes,counts,counts_sum,counts_sum_invr,norm_logits,logits_maxes,logits,h, hpreact,bnraw,bnvarr_inv,bngain,
        bnvarr,bndiff2,bndiff,hprebn,bnmean,embd_cat,embd]:
    t.retain_grad()
loss.backward()
loss


tensor(3.3038, grad_fn=<NegBackward0>)

In [18]:
def cmp(s,dt,t):
    ex = torch.all(dt == t.grad).item()
    app =torch.allclose(dt,t.grad)
    maxdiff =(dt-t.grad).abs().max().item()
    print(f'{s:15s} | exact:{str(ex):5s}|approximate:{str(app):5s} | maxdiff :{maxdiff}')
    

In [51]:
# exercise #1
# calculating derivative of each

#log probs
dlogprob = torch.zeros_like(logprob)
dlogprob[range(n),yb] = -1.0/n
cmp('logprob' ,dlogprob,logprob)
# probes 
dprobes = (1/probes)*dlogprob
cmp('prob' ,dprobes,probes)

dcounts_sum_invr = (counts*dprobes).sum(1,keepdim=True)
counts_sum_invr.shape

cmp('dcounts_sum_invr' ,dcounts_sum_invr,counts_sum_invr)

dcounts= (counts_sum_invr*dprobes)

dcount_sum = -counts_sum**-2*dcounts_sum_invr

dcounts+= torch.ones_like(counts) * dcount_sum

cmp('dcount_sum' ,dcount_sum,counts_sum)

cmp('dcounts' ,dcounts,counts)

dnorm_logits = norm_logits.exp()*dcounts
cmp('dnorm_logits' ,dnorm_logits,norm_logits)

dlogit  = dnorm_logits.clone()
dlogit_maxes = -dnorm_logits.sum(1,keepdim=True)

dlogit += F.one_hot(logits.max(1).indices,num_classes=logits.shape[1])*dlogit_maxes
cmp('dlogit' ,dlogit,logits)
cmp('dlogit_maxes' ,dlogit_maxes,logits_maxes)

dh = dlogit @ w2.T
cmp('dh' ,dh,h)
dw2 = h.T @ dlogit
cmp('dh' ,dw2,w2)

db2 = dlogit.sum(0,keepdim=True)
cmp('db2' ,db2,b2)


dhpreact = (1.0 - h**2) * dh

cmp('dhpreact' ,dhpreact,hpreact)

dbngain = ( bnraw * dhpreact ).sum(0 ,keepdim =True)
dbnraw = ( bngain * dhpreact )

cmp('dbngain', dbngain , bngain)
cmp('dbnraw', dbnraw , bnraw)

# bnraw = bndiff * bnvarr_inv

dbndiff = bnvarr_inv*dbnraw
dbnvarr_inv = (bndiff*dbnraw).sum(0,keepdim = True)

cmp('dbnvar_inv', dbnvarr_inv , bnvarr_inv)
# embd = C[xb]
# embd_cat = embd.view(embd.shape[0],-1)
# hprebn = embd_cat @ w1 +b1
# bnmean = 1/n * hprebn.sum(0,keepdim= True)
# bndiff = hprebn - bnmean 
# bndiff2 = bndiff**2
# bnvarr = 1/(n-1) * (bndiff2).sum(0,keepdim =True)
# bnvarr_inv = (bnvarr + 1e-15)**-0.5
# bnraw = bndiff * bnvarr_inv
# hpreact = bngain * bnraw + bnbias

dbnvarr = dbnvarr_inv  * -0.5 * (bnvarr + 1e-15)**-1.5
cmp('dbnvarr', dbnvarr , bnvarr)

dbndiff2 = dbnvarr * torch.ones_like(bndiff2) * 1/(n-1)
cmp('dbndiff2', dbndiff2 , bndiff2)

dbndiff += dbndiff2 * (2) * bndiff

cmp('dbndiff', dbndiff , bndiff)
dhprebn = dbndiff
dbnmean = -dbndiff.sum(0,keepdim=True)
dhprebn += dbnmean*1/n * (torch.ones_like(hprebn))


cmp('dhprebn', dhprebn , hprebn)
cmp('dbnmean', dbnmean , bnmean)

dembd_cat = dhprebn @ w1.T

cmp('dembd_cat', dembd_cat , embd_cat)

dw1 = embd_cat.T @ dhprebn

cmp('dw1', dw1 , w1)

db1 = dhprebn.sum(0,keepdim= True)
cmp('db1', db1 , b1)

dembd = dembd_cat.view(embd.shape)
cmp('dembd', dembd , embd)

dC = torch.zeros_like(C)
for i in range(xb.shape[0]):
    for j in range(xb.shape[1]):
        ix  = xb[i,j]
        dC[ix] += dembd[i,j]


cmp('dC', dC , C)






logprob         | exact:True |approximate:True  | maxdiff :0.0
prob            | exact:True |approximate:True  | maxdiff :0.0
dcounts_sum_invr | exact:True |approximate:True  | maxdiff :0.0
dcount_sum      | exact:True |approximate:True  | maxdiff :0.0
dcounts         | exact:True |approximate:True  | maxdiff :0.0
dnorm_logits    | exact:True |approximate:True  | maxdiff :0.0
dlogit          | exact:True |approximate:True  | maxdiff :0.0
dlogit_maxes    | exact:True |approximate:True  | maxdiff :0.0
dh              | exact:True |approximate:True  | maxdiff :0.0
dh              | exact:True |approximate:True  | maxdiff :0.0
db2             | exact:True |approximate:True  | maxdiff :0.0
dhpreact        | exact:False|approximate:True  | maxdiff :5.820766091346741e-11
dbngain         | exact:False|approximate:True  | maxdiff :2.3283064365386963e-10
dbnraw          | exact:False|approximate:True  | maxdiff :5.820766091346741e-11
dbnvar_inv      | exact:False|approximate:True  | maxdiff :3.4

In [53]:
loss_entropy = F.cross_entropy(logits,yb)
print(f'loss_diff : {(loss_entropy-loss).item()}')


loss_diff : 2.384185791015625e-07


In [54]:
dlogits = F.softmax(logits,1)
dlogits[range(n),yb] -=1
dlogits/=n
cmp("dlogits" ,dlogits,logits)


dlogits         | exact:False|approximate:True  | maxdiff :6.752088665962219e-09
